# Importação de Whisky

In [2]:
using JuMP
using GLPK
using Format

Sejam $I=\{1,2,3\}$ as garrafas importadas (Johnny Ballantine, Old Gargantua, Misty Deluxe) vendidas, e $V=\{1,2,3\}$ as garrafas vendidas (A, B, C).

In [3]:
I = collect(1:3);
V = collect(1:3);

Com variáveis $x_{iv}$ que definem o número de garrafas $i\in I$ importadas e usadas na mistura para $v\in V$

In [4]:
m = Model(GLPK.Optimizer)
@variable(m, x[i in I, v in V] >= 0);

preço de venda

In [5]:
pv = [68, 57, 45];

e custo de importação

In [6]:
ci = [70, 50, 40];

temos um lucro de

In [7]:
@variable(m, y[v in V]) ## total vendas
@variable(m, z[i in I]) ## total importação
@objective(m, Max, sum(pv[v] * y[v] for v in V) - sum(ci[i] * z[i] for i in I))

68 y[1] + 57 y[2] + 45 y[3] - 70 z[1] - 50 z[2] - 40 z[3]

In [8]:
@constraint(m, [v in V], y[v] == sum(x[i, v] for i in I))
@constraint(m, [i in I], z[i] == sum(x[i, v] for v in V));

Para definir os limites de importação

In [9]:
li = [2000, 2500, 1200];

vamos adicionar restrições

In [10]:
@constraint(m, [i in I], z[i] <= li[i]);

e para definir as restrições das percentagens

In [11]:
@constraint(m, x[1, 1] >= 0.6 * y[1]) ## no mínimo 60% Johnny Ballantine em A
@constraint(m, x[3, 1] <= 0.2 * y[1]) ## no máximo 20% Misty Deluxe em A

@constraint(m, x[1, 2] >= 0.15 * y[2]) ## no mínimo 15% Johnny Ballantine em B
@constraint(m, x[3, 2] <= 0.6 * y[2]) ## no máximo 60% Misty Deluxe em B

@constraint(m, x[3, 3] <= 0.6 * y[3]); ## no máximo 50% Misty Deluxe em C

Agora podemos mostrar o modelo todo e resolver

In [12]:
println(m)
optimize!(m)

Max 68 y[1] + 57 y[2] + 45 y[3] - 70 z[1] - 50 z[2] - 40 z[3]
Subject to
 -x[1,1] - x[2,1] - x[3,1] + y[1] = 0
 -x[1,2] - x[2,2] - x[3,2] + y[2] = 0
 -x[1,3] - x[2,3] - x[3,3] + y[3] = 0
 -x[1,1] - x[1,2] - x[1,3] + z[1] = 0
 -x[2,1] - x[2,2] - x[2,3] + z[2] = 0
 -x[3,1] - x[3,2] - x[3,3] + z[3] = 0
 x[1,1] - 0.6 y[1] ≥ 0
 x[1,2] - 0.15 y[2] ≥ 0
 z[1] ≤ 2000
 z[2] ≤ 2500
 z[3] ≤ 1200
 x[3,1] - 0.2 y[1] ≤ 0
 x[3,2] - 0.6 y[2] ≤ 0
 x[3,3] - 0.6 y[3] ≤ 0
 x[1,1] ≥ 0
 x[2,1] ≥ 0
 x[3,1] ≥ 0
 x[1,2] ≥ 0
 x[2,2] ≥ 0
 x[3,2] ≥ 0
 x[1,3] ≥ 0
 x[2,3] ≥ 0
 x[3,3] ≥ 0



Finalmente, vamos ver o lucro máximo

In [14]:
printfmtln("Importando {:.2f} garrafas de Johnny Ballantine, {:.2f} garrafas de Old Gargantua, e {:.2f} garrafas de Misty Deluxe, obtemos o lucro máximo de R\$ {:.2f} com as seguintes misturas:\n", value(z[1]), value(z[2]), value(z[3]), objective_value(m))
for v in V
	printfmt("Mistura {}: {:.2f} garrafas.\n", "ABC"[v], value(y[v]))
	if value(y[v]) > 0
		printfmt("  {:.2f}% JB, {:.2f}% OG, {:.2f}% MD.\n", "ABC"[v], value(x[1, v]) / value(y[v]), value(x[2, v]) / value(y[v]), value(x[3, v]) / value(y[v]))
	end
end

Importando 2000.00 garrafas de Johnny Ballantine, 2500.00 garrafas de Old Gargantua, e 1200.00 garrafas de Misty Deluxe, obtemos o lucro máximo de R$ 39888.89 com as seguintes misturas:

Mistura A: 2544.44 garrafas.
  65.00% JB, 0.60% OG, 0.20% MD.
Mistura B: 3155.56 garrafas.
  66.00% JB, 0.15% OG, 0.63% MD.
Mistura C: 0.00 garrafas.
